In [ ]:
!pip install shap polars optuna optuna-integration interpret

In [1]:
import polars as pl
from IPython.display import HTML

In [2]:
from src.pipeline import Pipeline, Context                                                                                                                                                                                                                                                             
from src.pipeline_steps import Load, Preprocess, TrainBinaryClassifier, TrainRegressor, TrainPerChannelShareRegressors, Predict, Report
from src.config import YEARLY_RECOVERY_DATA_CSV_FPS, S3_BUCKET_URI, S3_BUCKET, MODEL_DIR, CLF_MODEL_S3_KEY, REG_MODEL_S3_KEY, SHARE_MODELS_S3_PREFIX, REPORT_S3_KEY, ContextKeys

In [3]:
ctx = Context() 
Pipeline(
    Load(recovery_data_source=YEARLY_RECOVERY_DATA_CSV_FPS),
    Preprocess(write_to=f"{S3_BUCKET_URI}/data/processed/preprocessed_recovery_data.parquet"),
    TrainBinaryClassifier(read_from_key=CLF_MODEL_S3_KEY),
    TrainRegressor(read_from_key=REG_MODEL_S3_KEY),
    TrainPerChannelShareRegressors(save_to_prefix=SHARE_MODELS_S3_PREFIX),
    Predict(predict_channels=True, run_shap=False),
    Report(top_n=10, save_to=f"{S3_BUCKET_URI}/{REPORT_S3_KEY}")
)(ctx)

In [10]:
df_pre_data = ctx[ContextKeys.DF_RECOVERY_LOADED]

In [11]:
df_pre_data['year'].value_counts()

year,count
i64,u32
2022,10902782
2025,18981023
2023,11557004
2024,17414249


In [12]:
df_post_data = ctx['df_recovery_preprocessed']

In [13]:
df = ctx["predictions"]
df.head()

hashed_fc,gl_product_group,year,week,prob_recovered,p_nonzero,e_rate,combined_rate,abs_err,prob_donations,pred_prob_donations,abs_err_prob_donations,prob_liquidations,pred_prob_liquidations,abs_err_prob_liquidations,prob_return_to_vendor,pred_prob_return_to_vendor,abs_err_prob_return_to_vendor,prob_warehouse_deals_and_gr,pred_prob_warehouse_deals_and_gr,abs_err_prob_warehouse_deals_and_gr,prob_disposal,pred_prob_disposal,abs_err_prob_disposal,shap_baseline_rate,shap_deviation_contribution,rate_bucket,volume_bucket
str,f64,i64,f64,f64,f32,f32,f32,f64,f64,f32,f64,f64,f32,f64,f64,f32,f64,f64,f32,f64,f64,f32,f64,f64,f64,str,str
"""0006a543eceed85f71ee81511c9622…",60.0,2025,14.0,0.0,0.054249,0.999999,0.054249,0.054249,0.0,0.000351,0.000351,0.0,0.0208,0.0208,0.0,0.002807,0.002807,0.0,0.000197,0.000197,0.0,0.030094,0.030094,NaN,NaN,"""zero""","""<10"""
"""0006a543eceed85f71ee81511c9622…",60.0,2025,15.0,0.0,0.001767,0.999998,0.001767,0.001767,0.0,1.0000e-6,0.000001,0.0,0.000002,0.000002,0.0,0.001748,0.001748,0.0,0.0,0.0,0.0,0.000017,0.000017,NaN,NaN,"""zero""","""<10"""
"""0006a543eceed85f71ee81511c9622…",60.0,2025,16.0,0.0,0.005698,0.999999,0.005698,0.005698,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.005698,0.005698,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,"""zero""","""<10"""
"""0006a543eceed85f71ee81511c9622…",60.0,2025,22.0,0.0,0.057731,0.982793,0.056738,0.056738,0.0,0.000083,0.000083,0.0,0.048779,0.048779,0.0,0.000664,0.000664,0.0,0.000089,0.000089,0.0,0.007123,0.007123,NaN,NaN,"""zero""","""10-100"""
"""0006a543eceed85f71ee81511c9622…",60.0,2025,23.0,0.0,0.00045,0.493999,0.000223,0.000223,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000222,0.000222,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,"""zero""","""10-100"""


In [16]:
df.write_parquet(r's3://msds-26.2-data/out/predictions_2025.parquet')

In [14]:
df_post_data['year'].value_counts()

year,count
i64,u32
2025,2542216
2023,2069487
2024,2349233
2022,1807934


In [15]:
df['year'].value_counts()

year,count
i64,u32
2025,2542216
